In [21]:
class NodoCancion:
    def __init__(self, duracion , titulo):
        self.duracion = duracion
        self.titulo = titulo 
        self.izq = None
        self.der=None 

class MotorRecomendacionBST:
    def __init__(self):
        self.raiz=None
    
    def insertar(self , duracion , titulo):
        self.raiz= self._insertar_recursividad(self.raiz , duracion , titulo)

    def _insertar_recursividad(self , nodo_actual , duracion, titulo):
        if nodo_actual is None:
            return NodoCancion(duracion , titulo)
        if duracion < nodo_actual.duracion:
            nodo_actual.izq = self._insertar_recursividad(nodo_actual.izq, duracion, titulo)
        else:
            nodo_actual.der=self._insertar_recursividad(nodo_actual.der, duracion , titulo)
        return nodo_actual
    
    def calcular_tiempo_total(self , nodo_actual):
        if nodo_actual is None:
            return 0
        return nodo_actual.duracion + self.calcular_tiempo_total(nodo_actual.izq) + self.calcular_tiempo_total(nodo_actual.der)
    

    def recomendar_cancion(self , segundos_disponibles):
        return self._buscar_cercano(self.raiz , segundos_disponibles, None) 
    
    def _buscar_cercano(self, nodo_actual, objetivo, mejor_nodo):
        if nodo_actual is None:
            return mejor_nodo
        
        if mejor_nodo is None or abs(nodo_actual.duracion - objetivo) < abs (mejor_nodo.duracion - objetivo):
            mejor_nodo =nodo_actual

        if nodo_actual.duracion ==objetivo:
            return nodo_actual
        
        if objetivo < nodo_actual.duracion :
            return self._buscar_cercano(nodo_actual.izq , objetivo , mejor_nodo)
        else:
            return self._buscar_cercano(nodo_actual.der , objetivo , mejor_nodo)
        
    
    def filtrar_canciones_cortas(self , duracion_minima):
        self.raiz = self._filtrar_recursivo(self.raiz , duracion_minima)

    def _filtrar_recursivo(self , nodo_actual , limite):
        if nodo_actual is None:
            return None 
        
        nodo_actual.izq = self._filtrar_recursivo(nodo_actual.izq , limite)
        nodo_actual.der = self._filtrar_recursivo(nodo_actual.der , limite)

        if nodo_actual.duracion < limite:
            return nodo_actual.der
        
        return nodo_actual
    
    #4 Personalizada 
    def obtener_cancion_mas_larga(self):
        if self.raiz is None:
            return None
        return self._buscar_maximo(self.raiz)
    
    def _buscar_maximo(self , nodo_actual):
        if nodo_actual.der is None:
            return nodo_actual
        return self._buscar_maximo(nodo_actual.der)
    

In [22]:
import tkinter as tk
from tkinter import messagebox

arbol = MotorRecomendacionBST()

ventana= tk.Tk()

ventana.title("Motor de recomendacion de musica")
ventana.geometry("760x750")

canvas= tk.Canvas(ventana , width= 760 , height=250 , bg="white")
canvas.pack()

def dibujar_arbol():
    canvas.delete("all")
    if arbol.raiz:
        _dibujar(arbol.raiz , 375 , 30 , 140)

def _dibujar ( nodo , x , y, espacio):
    if nodo is None :
        return
    
    if nodo.izq:
        canvas.create_line(x , y , x - espacio, y +60 , fill ="green" , width=3)
        _dibujar(nodo.izq , x-espacio , y+60 , espacio // 2)

    if nodo.der:
        canvas.create_line(x , y , x + espacio, y + 60 , fill ="green" , width=3)
        _dibujar(nodo.der , x + espacio , y + 60 , espacio // 2)

    canvas.create_oval(x - 20 , y - 20 , x + 20 , y + 20, fill="blue", outline="yellow")
    canvas.create_text(x , y , text=str(nodo.duracion) , font=("Calibri", 11 , "bold"))
    canvas.create_text(x , y - 10 , text=str(nodo.titulo), font=("Calibri" , 10 , "italic"), fill= "black")

def agregar():
    try :
        titulo = entry_titulo.get().strip()
        duracion = int(entry_duracion.get())

        if not titulo : raise ValueError

        arbol.insertar(duracion , titulo)


        entry_titulo.delete(0 , tk. END)
        entry_duracion.delete(0 , tk.END)

        dibujar_arbol()
        lbl_info.config(text=f"Agregado: {titulo}({duracion})")

    except:
        messagebox.showerror( "Error" , "Intentelo de nuevo")

def tiempo_total():
    total = arbol.calcular_tiempo_total(arbol.raiz)
    messagebox.showinfo("Tiempo total es" , f"La playlist completa dura: {total} segundos ")

def recomendar():
    try:
        segundos= int(entry_duracion.get())
        nodo = arbol.recomendar_cancion(segundos)

        if nodo:
            messagebox.showinfo("Recomendacion" , f"Sugerencia optima: {nodo.titulo} ({nodo.duracion}s)")
        else :
            messagebox.showinfo("Recomendacion" , "El arbol se encuentra vacio")
    except:
        messagebox.showerror("Erro" , "Ingrese los segundos en el campo 'duracion' para recomendar")

def filtrar():
    try:
        min= int(entry_duracion.get())
        arbol.filtrar_canciones_cortas(min)

        dibujar_arbol()
        lbl_info.config(text= f"Canciones menores a {min}s eliminados")
    except:
        messagebox.showerror("Error", "Ingrese el limite 'duracion' para poder filtar")

def ver_max():
    nodo= arbol.obtener_canccion_mas_larga()
    if nodo :
        messagebox.showinfo("Cancion mas larga", f"Psita maxima : {nodo.titulo} ({nodo.duracion})")
    else:
        messagebox.showinfo("Cancion mas larga es", "No hay canciones registradas")
          


tk.Label(ventana , text="Titulo de la cancion" , font=("Calibri" , 12 , "bold")).pack(pady=2)
entry_titulo = tk.Entry(ventana , width= 40)
entry_titulo.pack(pady=2)

tk.Label(ventana , text="Duracion (segundos)" , font=("Calibri" , 12 , "bold")).pack(pady=2)
entry_duracion = tk.Entry(ventana , width= 40)
entry_duracion.pack(pady=2)


tk.Button(ventana , text="Agregar cancion" , command =agregar , width=25 , bg="green" , fg="white").pack(pady=4)
tk.Button(ventana , text="Tiempo total" , command = tiempo_total , width=25).pack(pady=4)
tk.Button(ventana , text="Recomendar" , command = recomendar , width=25).pack(pady=4)
tk.Button(ventana , text="Filtrar cortas" , command = filtrar , width=25 , bg="red" , fg="white").pack(pady=4)
tk.Button(ventana , text="Cancion mas larga" , command = ver_max , width=25).pack(pady=4)

lbl_info = tk.Label(ventana, text="Sistema de pistas" , font=("Calibri" , 10 , "italic") , fg="black")
lbl_info.pack(pady= 10)

ventana.mainloop()
